# 3.4 · 特征缩放 / Feature Scaling

> **课程定位 / Where this fits**
> **Part 3 第 4 课**。3.3 里 KNN/LOF/马氏都必须先 `StandardScaler`——这一课正式讲清**为什么、用哪种、怎么不泄漏**。一句话：**所有"算距离/算梯度"的模型都需要缩放，所有"按阈值切分"的树模型都不需要**。
> Distance/gradient-based models need scaling; threshold-splitting tree models don't.

> 💡 **面试相关 / Interview-relevant**
> - "哪些模型需要特征缩放" ★★★★★（必考，记住分界线）
> - "标准化 vs 归一化区别" ★★★★
> - "缩放怎么避免数据泄漏" ★★★★★
> - "有异常值时用什么缩放" ★★★（RobustScaler）

---

## 学习目标 / Learning Objectives
1. 记住**哪些模型需要 / 不需要**缩放的分界线（背后是"距离/梯度 vs 阈值"）。
2. 掌握 5 种 scaler：Standard / MinMax / Robust / MaxAbs / Quantile，各自适用。
3. 量化**不缩放对 KNN/SVM 的灾难性影响**。
4. 全程**防泄漏**：scaler 只 fit train。

## 目录 / TOC
1. [谁需要缩放, 谁不需要 ⭐](#1)
2. [🍷 数据：量纲差异灾难](#2)
3. [不缩放的灾难：KNN 实测](#3)
4. [五种 scaler 全家福 ⭐](#4)
5. [有异常值用 RobustScaler](#5)
6. [QuantileTransformer：强行正态化](#6)
7. [⚠ 防泄漏：scaler 只 fit train](#7)
8. [选择决策表](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 谁需要缩放, 谁不需要 ⭐ / Who Needs Scaling

**本质判据**：模型是否对"特征的数值大小"敏感。

| 需要缩放 ✅ | 原因 |
|---|---|
| KNN / K-Means | 算**欧氏距离**——大量纲特征主导 |
| SVM (RBF/linear) | 距离/核函数 |
| 线性/逻辑回归 + 正则化 | L1/L2 惩罚对系数大小敏感, 量纲不一惩罚不公 |
| 神经网络 | **梯度下降**收敛速度（不缩放则病态, 0.10 节）|
| PCA | 按**方差**找主成分, 大量纲特征假性主导 |

| 不需要缩放 ❌ | 原因 |
|---|---|
| 决策树 / 随机森林 / GBDT / XGBoost | 按**单特征阈值**切分, 单调变换不改变切分点 |
| 朴素贝叶斯 | 按列建分布 |

**一句话记忆**：**"算距离或算梯度的要缩放, 按阈值切分的不用"**。
"Distance or gradient → scale; threshold splits → don't."


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target
print(f"Wine: {X.shape}, {len(np.unique(y))} 类")
# 看量纲差异 / scale disparity
print("\n各特征的量级 (min-max):")
print(X.agg(["min","max","mean","std"]).T.round(2).head(8))


**量纲差异灾难**：`proline`（脯氨酸）范围 278-1680，`nonflavanoid_phenols` 范围 0.13-0.66——**差 3 个数量级**。在 KNN 里 proline 一个特征就主导了全部距离，其他 12 个特征形同虚设。
proline ranges in the thousands while some phenols are <1 — a 1000x disparity that lets one feature dominate all distances.


<a id="3"></a>
## 3. 不缩放的灾难：KNN 实测 / The Disaster, Measured

直接量化"不缩放 vs 缩放"对 KNN 准确率的影响。


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline

# 不缩放 / unscaled
acc_raw = cross_val_score(KNeighborsClassifier(), X, y, cv=5).mean()
# 缩放 (用 Pipeline 防泄漏, 3.12 节正题) / scaled, leak-free via pipeline
acc_scaled = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier()), X, y, cv=5).mean()

print(f"KNN 不缩放: 准确率 = {acc_raw:.1%}")
print(f"KNN 缩放后: 准确率 = {acc_scaled:.1%}")
print(f"提升 {(acc_scaled-acc_raw)*100:.0f} 个百分点 — 仅仅因为缩放!")

# 对比: 树模型不受影响 / trees are immune
from sklearn.ensemble import RandomForestClassifier
acc_rf_raw = cross_val_score(RandomForestClassifier(random_state=0), X, y, cv=5).mean()
acc_rf_scaled = cross_val_score(make_pipeline(StandardScaler(), RandomForestClassifier(random_state=0)), X, y, cv=5).mean()
print(f"\n随机森林 不缩放: {acc_rf_raw:.1%}  缩放后: {acc_rf_scaled:.1%}  (几乎不变 — 树对缩放免疫)")


**铁证**：同一份数据、同一个 KNN，**仅仅加上缩放，准确率从 ~70% 跳到 ~95%**。而随机森林缩放前后几乎一样——完美印证第 1 节的分界线。
Same KNN, same data — scaling alone jumps accuracy ~25 points. Random forest is unmoved. The dividing line holds.


<a id="4"></a>
## 4. 五种 scaler 全家福 ⭐ / The Five Scalers

| Scaler | 公式 | 输出范围 | 对异常值 |
|---|---|---|---|
| **StandardScaler** | $(x-\mu)/\sigma$ | 均值 0 方差 1, 无界 | 敏感 (用 μ,σ) |
| **MinMaxScaler** | $(x-\min)/(\max-\min)$ | $[0,1]$ | **极敏感** (min/max 被异常值定义) |
| **RobustScaler** | $(x-\text{median})/\text{IQR}$ | 无界, 中位数 0 | **稳健** ⭐ |
| **MaxAbsScaler** | $x/\max(\lvert x\rvert)$ | $[-1,1]$ | 敏感; 保留稀疏性(不平移) |
| **QuantileTransformer** | 映射到分位数 | $[0,1]$ 或正态 | 稳健, 但非线性 |


In [ ]:
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   MaxAbsScaler, QuantileTransformer)

col = X["proline"].values.reshape(-1, 1)
scalers = {
    "原始": col.ravel(),
    "Standard": StandardScaler().fit_transform(col).ravel(),
    "MinMax": MinMaxScaler().fit_transform(col).ravel(),
    "Robust": RobustScaler().fit_transform(col).ravel(),
    "MaxAbs": MaxAbsScaler().fit_transform(col).ravel(),
}
fig, axes = plt.subplots(1, 5, figsize=(15, 2.8))
for ax, (name, v) in zip(axes, scalers.items()):
    ax.hist(v, bins=30); ax.set_title(f"{name}\n[{v.min():.1f}, {v.max():.1f}]", fontsize=9)
plt.suptitle("proline 经各 scaler 后 — 形状不变, 只是位置/尺度变了 (线性缩放)", y=1.08)
plt.tight_layout(); plt.show()
print("注意: Standard/MinMax/Robust/MaxAbs 都是线性变换 → 分布形状不变, 只改位置和尺度")


<a id="5"></a>
## 5. 有异常值用 RobustScaler / RobustScaler for Outliers

**MinMax 的致命弱点**：用 min/max 定义范围 → **一个极端异常值就把所有正常数据挤进一个小角落**。RobustScaler 用中位数/IQR，免疫。


In [ ]:
# 注入一个异常值, 对比 MinMax vs Robust / inject an outlier
data = np.append(rng.normal(50, 10, 200), [500])  # 一个 500 的异常 / one outlier at 500
data = data.reshape(-1, 1)

mm = MinMaxScaler().fit_transform(data).ravel()
rb = RobustScaler().fit_transform(data).ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(mm[:-1], bins=40); axes[0].axvline(mm[-1], color="r", label="outlier")
axes[0].set_title(f"MinMax: 正常数据全挤在 [0, {mm[:-1].max():.2f}]\n异常值霸占了整个 [0,1] 空间")
axes[0].legend()
axes[1].hist(rb[:-1], bins=40); axes[1].axvline(rb[-1], color="r", label="outlier")
axes[1].set_title(f"Robust: 正常数据正常分布\n异常值 z={rb[-1]:.0f} 留在远处不挤压主体")
axes[1].legend()
plt.tight_layout(); plt.show()
print("MinMax 下正常数据被压到极小区间, 模型几乎分不开 — 有异常值时禁用 MinMax")


<a id="6"></a>
## 6. QuantileTransformer：强行正态化 / Forcing Normality

前 4 种是**线性**变换（不改形状）。`QuantileTransformer` 是**非线性**的——把任意分布**强行掰成均匀或正态**。

**代价与收益**：
- ✅ 极右偏数据（收入/fare）一键正态化, 对假设正态的模型友好
- ✅ 天然压缩异常值（映射到分位数）
- ❌ **非线性扭曲**了原始关系, 可解释性下降; 小样本下分位数估计不稳


In [ ]:
# 把极右偏数据正态化 / normalize a heavily skewed feature
skewed = rng.lognormal(0, 1, 1000).reshape(-1, 1)
qt = QuantileTransformer(output_distribution="normal", random_state=0)
normalized = qt.fit_transform(skewed).ravel()

import scipy.stats as st
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(skewed, bins=50); axes[0].set_title(f"原始 lognormal (skew={st.skew(skewed.ravel()):.2f})")
axes[1].hist(normalized, bins=50); axes[1].set_title(f"QuantileTransformer→normal (skew={st.skew(normalized):.2f})")
plt.tight_layout(); plt.show()
print("强行正态化 — 但记住这是非线性变换, 改变了数据间的相对关系")


<a id="7"></a>
## 7. ⚠ 防泄漏：scaler 只 fit train / Leak-free Scaling

和 3.2 填补一样的铁律, 但缩放更隐蔽更常错：

✅ **正确流程**：
1. 划分 train/test
2. `scaler.fit(X_train)` — 学到 train 的 μ, σ（或 min/max/median）
3. `scaler.transform(X_train)` 和 `scaler.transform(X_test)` — **test 用 train 的统计量**

❌ **常见错误**：`scaler.fit_transform(X)` 在划分前对全数据做——test 的均值/方差渗入训练。


In [ ]:
from sklearn.model_selection import train_test_split

X_tr, X_te = train_test_split(X[["proline"]], test_size=0.3, random_state=0)

# ❌ 错: fit 全数据 / WRONG
wrong_scaler = StandardScaler().fit(X[["proline"]])
# ✅ 对: fit 只 train / RIGHT
right_scaler = StandardScaler().fit(X_tr)

print(f"全数据 proline 均值 (错误): {wrong_scaler.mean_[0]:.1f}, std: {wrong_scaler.scale_[0]:.1f}")
print(f"仅 train  proline 均值 (正确): {right_scaler.mean_[0]:.1f}, std: {right_scaler.scale_[0]:.1f}")
print("\ntest 集必须用 train 的 (μ,σ) 来 transform:")
X_te_scaled = right_scaler.transform(X_te)
print(f"test 缩放后均值 = {X_te_scaled.mean():.3f} (不一定是 0! 因为用的是 train 的 μ — 这才对)")
print("\n💡 test 缩放后均值 ≈ 0 但不精确等于 0, 正是没泄漏的标志")


<a id="8"></a>
## 8. 选择决策表 / Decision Table

```
模型是树/森林/GBDT/朴素贝叶斯? → 不用缩放, 跳过
否则需要缩放, 选哪个:
  数据近正态, 无严重异常     → StandardScaler (默认首选)
  需要固定 [0,1] (如图像/神经网络输入) → MinMaxScaler (但先确认无异常值)
  有异常值                  → RobustScaler ⭐
  稀疏数据 (大量0, 如词频)   → MaxAbsScaler (不破坏稀疏性)
  极偏态 + 模型要正态        → QuantileTransformer / PowerTransformer
永远: scaler.fit() 只在 train, 用 Pipeline (3.12) 自动保证
```

> 💡 还有 `PowerTransformer`（Box-Cox / Yeo-Johnson）——参数化地把数据变正态, 比 Quantile 更平滑可逆, 偏态数据值得一试。
> PowerTransformer (Box-Cox/Yeo-Johnson) is a smoother, invertible alternative for skew correction.


<a id="9"></a>
## 9. 小结 / Summary

```
谁要缩放: 距离(KNN/SVM/KMeans) + 梯度(NN) + 正则(线性) + PCA ✅
谁不要:   树/森林/GBDT/朴素贝叶斯 (按阈值切分) ❌
实测: KNN 加缩放 70%→95%, 随机森林不变

五种 scaler:
  Standard (默认) | MinMax (固定[0,1],怕异常值) | Robust (有异常值用) ⭐
  MaxAbs (保稀疏) | Quantile (非线性强行正态)
防泄漏: scaler.fit() 只在 train
```

### 💡 面试速查
1. **分界线**："算距离/梯度要缩放, 按阈值切分不用"
2. **树模型不需要缩放**（单调变换不改切分点）
3. **有异常值用 RobustScaler**（MinMax 会被异常值绑架）
4. **scaler 只 fit train** — 否则泄漏

### 下一节
**3.5 类别变量编码**——数值特征处理完了, 类别特征（sex/country/plan）怎么变成数字？label/one-hot/target/frequency 编码。
